In [4]:
from __future__ import annotations

import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional

import mesa
import numpy as np

from lib.datagen.samplers import Pi, message_sampler, spin_sampler


class OpinionAgent(mesa.Agent):
    """An LLM-driven agent with a persona, opinion, and message inboxes."""

    def __init__(
        self,
        model,
        agent_index: int,
        persona: str,
    ):
        super().__init__(unique_id=agent_index, model=model)

        self.agent_index = agent_index
        self.persona = persona

        self.spin = 0
        self.raw_spins: list[int] = []

        self.inbox_agree: list[str] = []
        self.inbox_disagree: list[str] = []

        self.last_message: Optional[str] = None

        # Per-agent histories.
        self.spin_history: list[int] = []
        self.message_history: list[str] = []

    def sample_spin(self) -> int:
        """Sample the agent's opinion using its current inboxes."""
        return self.model.sample_agent_spin(self)

    def generate_message(self) -> str:
        """Generate a message from the agent's current state."""
        return self.model.generate_agent_message(self)

    def step(self):
        """
        Mesa agent activation is intentionally not used to update opinions.

        The model coordinates synchronized message and spin phases.
        """
        pass

In [5]:
class OpinionModel(mesa.Model):
    """
    Synchronous, LLM-driven opinion dynamics.

    Each call to step():
      - First call: initialize opinions with empty inboxes.
      - Subsequent calls: generate messages, deliver them, then
        update opinions.

    J[i, j] determines whether sender i communicates with receiver j
    and which inbox receives the message.
    """

    def __init__(
        self,
        personas: list[str],
        statement: str,
        J: np.ndarray,
        pi: Pi,
        *,
        name: str = "replica_0",
        mode: str = "subjective",
        choices: Optional[dict[str, str]] = None,
        k: int = 5,
        max_workers: int = 64,
        seed: Optional[int] = None,
    ):
        super().__init__(seed=seed)

        self.name = name
        self.personas = list(personas)
        self.statement = statement
        self.J = np.asarray(J, dtype=np.int8).copy()
        self.pi = pi

        self.mode = mode
        self.choices = choices
        self.k = k
        self.max_workers = max_workers

        self.n = len(self.personas)

        if self.J.shape != (self.n, self.n):
            raise ValueError(
                f"J has shape {self.J.shape}, "
                f"expected {(self.n, self.n)}"
            )

        if not np.isin(self.J, [-1, 0, 1]).all():
            raise ValueError("J must contain only -1, 0, +1.")

        if self.k < 1:
            raise ValueError("k must be >= 1.")

        if self.mode not in {"subjective", "objective"}:
            raise ValueError(
                "mode must be 'subjective' or 'objective'."
            )

        if self.mode == "objective":
            if not choices or not {"A", "B"} <= choices.keys():
                raise ValueError(
                    "Objective mode requires choices for A and B."
                )

        # Agents are registered with Mesa automatically.
        self.opinion_agents = [
            OpinionAgent(self, i, persona)
            for i, persona in enumerate(self.personas)
        ]

        # Model-level histories.
        # spins_history[t] has shape (n,).
        # spins_raw_history[t] has shape (n, k).
        # messages_history[t] maps (sender, receiver) -> message.
        self.spins_history: list[np.ndarray] = []
        self.spins_raw_history: list[np.ndarray] = []
        self.messages_history: list[dict[tuple[int, int], str]] = []

        self._initialized = False

        # One executor reused across all rounds.
        self.executor = ThreadPoolExecutor(
            max_workers=self.max_workers
        )

    def sample_agent_spin(self, agent: OpinionAgent) -> int:
        """One spin-sampler call, with retries for ValueError."""
        for _ in range(3):
            try:
                result = spin_sampler(
                    agent.persona,
                    self.statement,
                    agent.inbox_agree,
                    agent.inbox_disagree,
                    self.pi,
                    mode=self.mode,
                    choices=self.choices,
                )

                result = int(result)

                if result not in (-1, 0, 1):
                    raise ValueError(
                        f"Invalid spin returned: {result}"
                    )

                return result

            except ValueError:
                continue

        return 0

    def generate_agent_message(
        self,
        agent: OpinionAgent,
    ) -> str:
        """Generate one message for the sender's outgoing edges."""
        return message_sampler(
            persona=agent.persona,
            question=self.statement,
            messages_agree=list(agent.inbox_agree),
            messages_disagree=list(agent.inbox_disagree),
            current_spin=agent.spin,
            pi=self.pi,
            mode=self.mode,
            choices=self.choices,
        )

    @staticmethod
    def _aggregate_spin(samples: list[int]) -> int:
        """Majority vote; resolve ties using the first sample."""
        total = sum(samples)

        if total > 0:
            return 1
        if total < 0:
            return -1

        return samples[0] if samples else 0

    def _update_spins(self, step_idx: int) -> None:
        """Sample k spins per agent and update all states together."""
        futures = {}

        for agent in self.opinion_agents:
            for sample_idx in range(self.k):
                future = self.executor.submit(
                    self.sample_agent_spin,
                    agent,
                )
                futures[future] = (
                    agent.agent_index,
                    sample_idx,
                )

        raw = np.zeros((self.n, self.k), dtype=np.int8)

        for future in as_completed(futures):
            agent_idx, sample_idx = futures[future]

            try:
                raw[agent_idx, sample_idx] = future.result()
            except Exception:
                raw[agent_idx, sample_idx] = 0

        # Commit state only after every agent has been sampled.
        spins = np.array(
            [
                self._aggregate_spin(raw[i].tolist())
                for i in range(self.n)
            ],
            dtype=np.int8,
        )

        for i, agent in enumerate(self.opinion_agents):
            agent.raw_spins = raw[i].tolist()
            agent.spin = int(spins[i])
            agent.spin_history.append(int(spins[i]))

        self.spins_raw_history.append(raw.copy())
        self.spins_history.append(spins.copy())

    def _communicate(self, step_idx: int) -> None:
        """Generate one message per sender, then broadcast it."""
        futures = {}

        # Determine each sender's outgoing recipients.
        recipients: dict[int, list[int]] = {}

        for i in range(self.n):
            recipients[i] = np.flatnonzero(
                self.J[i] != 0
            ).tolist()

        # Only agents with outgoing edges need to generate messages.
        for i, js in recipients.items():
            if not js:
                continue

            agent = self.opinion_agents[i]

            future = self.executor.submit(
                self.generate_agent_message,
                agent,
            )
            futures[future] = i

        sender_messages: dict[int, str] = {}

        for future in as_completed(futures):
            sender_idx = futures[future]

            try:
                sender_messages[sender_idx] = future.result()
            except Exception as exc:
                # Preserve the behavior of the original implementation.
                sender_messages[sender_idx] = f"[error: {exc}]"

        # Construct the complete message set before delivering anything.
        step_messages: dict[tuple[int, int], str] = {}

        for i, js in recipients.items():
            if not js:
                continue

            message = sender_messages[i]

            for j in js:
                step_messages[(i, j)] = message

            self.opinion_agents[i].last_message = message
            self.opinion_agents[i].message_history.append(message)

        # Build fresh inboxes. No receiver is updated during generation.
        agree = [[] for _ in range(self.n)]
        disagree = [[] for _ in range(self.n)]

        incoming: dict[int, list[tuple[int, str]]] = {}

        for (i, j), message in sorted(step_messages.items()):
            incoming.setdefault(j, []).append((i, message))

        for j, items in incoming.items():
            # Stable, reproducible shuffle independent of task completion
            # order and the global random-number generator.
            random.Random(
                f"{self.name}|{step_idx}|{j}"
            ).shuffle(items)

            for i, message in items:
                if self.J[i, j] == 1:
                    agree[j].append(message)
                elif self.J[i, j] == -1:
                    disagree[j].append(message)

        for i, agent in enumerate(self.opinion_agents):
            agent.inbox_agree = agree[i]
            agent.inbox_disagree = disagree[i]

        self.messages_history.append(
            dict(sorted(step_messages.items()))
        )

    def step(self) -> None:
        """Advance the simulation by one synchronized transition."""
        if not self._initialized:
            # Initial opinions, before any messages have been exchanged.
            self._update_spins(step_idx=0)
            self._initialized = True
            return

        # Round t: messages from the previous state, followed by spins.
        step_idx = len(self.spins_history)

        self._communicate(step_idx)
        self._update_spins(step_idx)

    def run(self, num_steps: int = 8) -> "OpinionModel":
        """
        Run num_steps communication rounds.

        Produces num_steps + 1 opinion snapshots, including t=0.
        """
        if num_steps < 0:
            raise ValueError("num_steps must be >= 0.")

        if not self._initialized:
            self.step()

        for _ in range(num_steps):
            self.step()

        return self

    def save(self, output_dir: str | Path) -> Path:
        """Save model state and histories to one JSON file."""
        import json

        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        payload: dict[str, Any] = {
            "meta": {
                "name": self.name,
                "n": self.n,
                "num_steps": len(self.spins_history) - 1,
                "k": self.k,
                "mode": self.mode,
            },
            "personas": self.personas,
            "statement": self.statement,
            "choices": self.choices,
            "J": self.J.astype(int).tolist(),
            "spins_history": [
                s.astype(int).tolist()
                for s in self.spins_history
            ],
            "spins_raw_history": [
                s.astype(int).tolist()
                for s in self.spins_raw_history
            ],
            "messages_history": [
                {
                    f"{i},{j}": message
                    for (i, j), message in messages.items()
                }
                for messages in self.messages_history
            ],
        }

        output_path = out_dir / f"{self.name}.json"

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        return output_path

    def close(self) -> None:
        """Release the thread pool when the simulation is finished."""
        self.executor.shutdown(wait=True)

In [ ]:
from lib.datagen.utils import make_gemini_pi

personas = [
    "You are a cautious scientist who values empirical evidence.",
    "You are an environmental activist concerned about climate change.",
    "You are an economist who prioritizes economic growth.",
    "You are a skeptical citizen who questions mainstream narratives.",
]

statement = "Governments should prioritize climate policies."

# Directed, signed interaction network.
#
# J[i, j] = +1: i's message goes into j's agreement inbox.
# J[i, j] = -1: i's message goes into j's disagreement inbox.
# J[i, j] =  0: no communication.
J = np.array(
    [
        [0,  1, -1,  0],
        [1,  0,  1, -1],
        [-1, 1,  0,  1],
        [0, -1,  1,  0],
    ],
    dtype=np.int8,
)

# Configure this using your existing Pi implementation.
pi = make_gemini_pi(model="gemini-3.5-flash-lite")

model = OpinionModel(
    personas=personas,
    statement=statement,
    J=J,
    pi=pi,
    name="climate_replica_0",
    mode="subjective",
    k=5,
    max_workers=16,
    seed=42,
)

try:
    model.run(num_steps=8)

    print("Final opinions:")

    for agent in model.opinion_agents:
        print(
            f"Agent {agent.agent_index}: "
            f"spin={agent.spin}, "
            f"agree_messages={len(agent.inbox_agree)}, "
            f"disagree_messages={len(agent.inbox_disagree)}"
        )

    path = model.save("personal/mesa/outputs")
    print(f"Saved results to {path}")

finally:
    model.close()

Final opinions:
Agent 0: spin=1, agree_messages=1, disagree_messages=1
Agent 1: spin=1, agree_messages=2, disagree_messages=1
Agent 2: spin=-1, agree_messages=2, disagree_messages=1
Agent 3: spin=-1, agree_messages=1, disagree_messages=1
Saved results to outputs/climate_replica_0.json
